# Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
import numpy as np
import torch
from scipy.ndimage import binary_dilation
import matplotlib.pyplot as plt
import cv2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device type {device.type}")

# Loading experiment data

In [ ]:
# Modify these to load experiment data from a directory
data_dir = "../../reference2/MSH2-2/EVsint_tiff/"
name = "POLE1_ASYNC"

In [ ]:
from experiment_loader import ExperimentBatch
batch = ExperimentBatch(name, data_dir)
print(f"Number of experiments in batch: {len(batch.experiments())}")

## Calculate minumum number of frames for the experiments in the batch

In [ ]:
lens = []
for experiment in batch.experiments():
    lens.append(len(experiment.frames_u8()))
min_len = min(lens)
max_len = max(lens)
batch.max_len = min_len
print(f"Minimum number of frames {min_len}")
print(f"Maximum number of frames {max_len}")

# ROI Editor

Interactive editor for correcting ROI boundaries. Drag the black lines up/down to adjust ranges; changes are saved automatically next to the original ROI file.

In [ ]:
%matplotlib widget
from roi_editor import show_roi_editors
show_roi_editors(batch.experiments())

# Generating cell masks

## Config

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport image_transforms

In [ ]:
from image_transformer import generate_masks_using_filters_and_transforms
from image_transforms import convert_to_u8, threshold_image, adjust_contrast, set_brightness

In [ ]:
# Filter which experiments from the batch are processed. This can speed up iteration time while tuning the filters
# Possible values:
#     None             all experiments
#     list of numbers  [0, 2, 4] only these experiments
#     string           all experiments containing the string. E.g. 'KO' will show all experiments whose file name includes 'KO'
experiment_filter = [0]
# Which frames to process from each experiment. 
# Possible values:
#     integer           1, 2, 3, etc. Evenly split the experiment interval with N points.
#                       1 will show the first frame, 2 shows the first and last frames, 3 shows first, middle, last
#     list of integers  Show the frames specified in the list, e.g. [0, 10] shows the 0th and 10th frames
frame_filter = 2
# Additionally to the final result, show the processed frames after applying these tranforms. It must be a list of integers.
# Can be indexed with negative numbers to show the result after the last transform
show_after_transform = [0, 2,3]
# The list of transforms to perform on each image before handing it to the masking algorithm.
# You can find a list of transforms with examples in transforms.ipynb
transforms = [
    # Identity transform. With 0 in show_after_transform this will show the original images.
    lambda frame: frame,
    # Convert the data format into uint8. This must be done before genereating the masks. The masking algorithm cannot handle other formats 
    convert_to_u8,
    set_brightness(20),
    threshold_image(20),
]

generate_masks_using_filters_and_transforms(batch, experiment_filter, frame_filter, transforms, show_after_transform)

# Cell video generation 

This part applies the transforms to all experiments in the loaded batch and runs mask generation for each frame. This can take some time. The results are saved to disk. The later stages use this data from the disk to plot the results 

In [ ]:
from experiment_evaluator import evalute_batch
evalute_batch(batch, transforms=transforms)

# Cell video outlier filtering

In [ ]:
# load segmentation results from disk

from experiment_evaluator import ExperimentEvaluator

vids = []
for experiment in batch.experiments():
    print(f"Loading experiment {experiment.name}")
    exp = ExperimentEvaluator.load_experiment(experiment)
    vids += exp.cell_videos()


In [ ]:
from cell_video_visualizer import normalize_videos_for_display

# Sets the size of the ROI range
range_modifier = 5
vids2 = normalize_videos_for_display(vids, range_modifier=range_modifier)

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport cell_video_visualizer

In [ ]:
# These set the checkboxes
manual_outliers = set()

In [ ]:
from cell_video_visualizer import show_video_grid_jupyter
batch_size = 30
updaters = []
for i in range(0, len(vids2), batch_size):
    updaters.append(show_video_grid_jupyter(vids2[i:i+batch_size], manual_outliers=manual_outliers))
update_outliers = lambda outliers, manual: [updater(outliers, manual) for updater in updaters]

In [ ]:
manual_outliers

In [ ]:
from outlier_filters import area_between, max_area_change_between_frames, max_total_area_change, id_starts_with
from outlier_filters import outlier_vids

# copy the ids of cells here to always set them as outliers. For example {"REV7_WT_1700_001_24"}
extra_manual_outliers = {'POLE1_KO_1700_001_4'}

filters = [
    area_between(10, 5000),
    #max_area_change_between_frames(0.2),
    #max_total_area_change(0.2),
    #id_starts_with('REV'),
]
outliers = outlier_vids(vids, extra_manual_outliers, filters)
_ = update_outliers(outliers, extra_manual_outliers)

# Data evaluation and visualization

In [ ]:
cell_filter = lambda vid: vid.id not in outliers and vid.id not in manual_outliers

In [ ]:
from experiment_loader import ExperimentType
from experiment_evaluator import ScoreType, plot_brightness_score_double, plot_brightness_score

plot_brightness_score(batch, score_type=ScoreType.Ratio, range_modifier=range_modifier, cell_filter=cell_filter)

In [ ]:
from video_exporter import export_cell_videos
export_cell_videos(vids, os.path.join(data_dir, "brightness_scores.csv"), range_modifier=range_modifier)

In [ ]:
from experiment_evaluator import plot_all_vid_scores
from experiment_evaluator import calculate_brightness_score_for_videos, ExperimentType, ScoreType
ids, scores = calculate_brightness_score_for_videos(vids, score_type=ScoreType.Diff, range_modifier=range_modifier, cell_filter=cell_filter)
plot_all_vid_scores(scores, ids)